# Corpus construction

From the filtered corpus to the corpus the model counts. Writes `data/interim/02_corpus.csv`, one row per version of a file.

- **Repeats:** a passage repeated right after itself enters once.
- **Weights:** versions of one piece share one weight.
- **Groups and folds:** related files share a group; groups go to cross-validation folds.

In [ ]:
import os, json
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
from music21 import converter

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")   # written by one notebook, read by the next
WORKERS = os.cpu_count() // 2
NEAR = 7.4   # % of differing notes up to which two passages are versions of one passage (set by the gap in the copies below)

table = pd.read_csv(INTERIM / "01_corpus.csv").set_index("file")   # one row per file, from 01_filtering

def in_parallel(function, args, chunksize=8):   # one call per argument, results in the same order
    with ProcessPoolExecutor(max_workers=WORKERS) as pool:
        return list(pool.map(function, args, chunksize=chunksize))

def lines_of(file):   # per voice, (onset, duration, pitch) of each note, tied notes merged; the top voice first
    score = converter.parse(RAW / table.at[file, "corpus"] / file)
    return [[(float(n.offset), float(n.quarterLength), max(p.midi for p in n.pitches)) for n in score.parts[int(k)].stripTies().flatten().notes]
            for k in table.at[file, "voice_parts"].split("|")]

def matched(a, b):   # notes of a found in b, in order
    return sum(block.size for block in SequenceMatcher(None, a, b, autojunk=False).get_matching_blocks())

lines = dict(zip(table.index, in_parallel(lines_of, table.index)))
table.groupby("corpus").size().rename("files").to_frame()

## Passages repeated inside a file

- **Repeat signs:** not expanded.
- **Written-out repeat:** at least eight notes of the top voice, same pitches and durations, again right after themselves; every voice compared.
- **Identical:** the second occurrence is skipped.
- **Differ by at most 7.4%:** two versions; the file enters twice, without each occurrence, with weight one half each.
- **Differ more, or come back later:** stay, as different music or composed form.

In [ ]:
def repeats_of(file):   # adjacent repeats of at least eight top-voice notes, and how much the voices differ between the two occurrences
    voices = lines[file]
    top = [(d, p) for _, d, p in voices[0]]
    found = []
    for i in range(len(top)):
        for j in range(i + 1, len(top)):
            if i > 0 and top[i - 1] == top[j - 1]:
                continue   # not the start of the repeated passage
            n = 0
            while j + n < len(top) and i + n < j and top[i + n] == top[j + n]:
                n += 1
            if n < 8 or j != i + n:
                continue   # too short, or not right after itself
            start, again = voices[0][i][0], voices[0][j][0]
            def block(line, at): return [(o - at, d, p) for o, d, p in line if at <= o < at + again - start]
            first, second = [block(line, start) for line in voices], [block(line, again) for line in voices]
            short = sum(min(len(a), len(b)) for a, b in zip(first, second))
            differ = short - sum(matched(a, b) for a, b in zip(first, second))   # notes of the shorter occurrence the other lacks
            found.append({"file": file, "start": start, "again": again, "end": 2 * again - start, "top notes": n,
                          "identical": first == second, "differ %": round(100 * differ / short, 1)})
    return found

repeats = pd.DataFrame([r for rows in in_parallel(repeats_of, table.index, chunksize=4) for r in rows])
repeats["corpus"] = repeats["file"].map(table["corpus"])
repeats["kind"] = np.select([repeats["identical"], repeats["differ %"] <= NEAR], ["identical", "versions"], "different")
assert (repeats[repeats["kind"] == "versions"].groupby("file").size() == 1).all()   # at most one pair of versions per file
display(repeats.groupby(["corpus", "kind"]).size().unstack(fill_value=0))
repeats[repeats["kind"] == "versions"].sort_values("differ %").drop(columns=["identical", "kind"])

**Result**
- Bach: 26 identical written-out repeats, 7 with small differences (0% to 2.7%).
- Palestrina: no identical one, 2 with small differences (0% and 3.2%). A difference of 0% means the shorter occurrence lies inside the longer.
- The other adjacent repeats, 26 in Bach and 10 in Palestrina, differ more and stay.

In [ ]:
skips = defaultdict(list)   # file -> time spans (start, end) in quarter notes to leave out, for every version
for r in repeats[repeats["kind"] == "identical"].itertuples():
    skips[r.file].append((r.again, r.end))
versions = {}   # file -> the spans each of its two versions leaves out
for r in repeats[repeats["kind"] == "versions"].itertuples():
    versions[r.file] = [skips[r.file] + [(r.again, r.end)], skips[r.file] + [(r.start, r.again)]]

def collapsed(file):   # the voices of the first version: notes inside a skipped span leave
    spans = versions[file][0] if file in versions else skips[file]
    return [[(o, d, p) for o, d, p in line if not any(s <= o < e for s, e in spans)] for line in lines[file]]

kept_notes = sum(len(n) for f in table.index for n in collapsed(f))
print(f"Notes in the voices: {sum(len(n) for v in lines.values() for n in v):,}; in the first version of every file: {kept_notes:,}.")

## Weights

- **Rule:** each version of a piece (same `copy_group`) weighs 1/k, with k versions, so the notes they share count once.
- **Source:** sequence weighting, [Henikoff & Henikoff 1994](https://doi.org/10.1016/0022-2836(94)90032-9).

In [ ]:
weight = 1 / table.groupby("copy_group")["copy_group"].transform("size")   # versions of one piece share its weight
print(f"Pieces with more than one version: {(table['copy_group'].value_counts() > 1).sum()}, in {(weight < 1).sum()} files.")

## Groups

- **Why:** a model must never train on the piece it is tested on.
- **Bach, the tune:** the Zahn number that When in Rome gives each chorale, the melody's number in [Zahn (1889–1893)](https://cpdl.org/wiki/index.php/Die_Melodien_der_deutschen_evangelischen_Kirchenlieder_(Johannes_Zahn)); also top voices that share 90% of their intervals, which covers the chorales without an analysis.
- **Palestrina, the mass:** its movements share modes, motifs and often a model.
- Versions and copies always share a group.

In [ ]:
def record(file, tag):   # a reference record of a kern file
    return next(line.split(":", 1)[1].strip() for line in open(RAW / "palestrina" / file, encoding="latin-1") if line.startswith(f"!!!{tag}:"))
parent = {f: f for f in table.index}   # files joined into groups
def root(f):
    while parent[f] != f:
        f = parent[f]
    return f
for members in table.groupby("copy_group").groups.values():   # versions of one piece
    for f in members[1:]:
        parent[root(f)] = root(members[0])

ANALYSES = RAW / "when_in_rome"
zahn, tune = {}, {}
for folder in sorted(ANALYSES.iterdir()):   # the lowest Riemenschneider number that names the file
    remote = json.load(open(folder / "remote.json"))
    for name in remote.get("music21") or []:
        zahn.setdefault(name, remote.get("Zahn"))
        tune.setdefault(name, remote.get("tune"))

def melodies_close(i):   # chorales whose top-voice intervals contain each other's at 90% or more
    a = MELODY[BACH[i]]
    return [(BACH[i], BACH[j]) for j in range(i + 1, len(BACH))
            if matched(a, MELODY[BACH[j]]) >= 0.9 * min(len(a), len(MELODY[BACH[j]]))]
BACH = [f for f in table.index if table.at[f, "corpus"] == "bach"]
MELODY = {f: [b[2] - a[2] for a, b in zip(lines[f][0], lines[f][0][1:])] for f in BACH}
close = [p for rows in in_parallel(melodies_close, range(len(BACH)), chunksize=4) for p in rows]

for a, b in close:
    parent[root(a)] = root(b)
by_zahn = defaultdict(list)
for f in BACH:
    if zahn.get(f):
        by_zahn[zahn[f]].append(f)
for members in by_zahn.values():
    for f in members[1:]:
        parent[root(f)] = root(members[0])
masses = defaultdict(list)
for f in table.index[table["corpus"] == "palestrina"]:
    masses[record(f, "OPR")].append(f)
for members in masses.values():
    for f in members[1:]:
        parent[root(f)] = root(members[0])

roots = [root(f) for f in table.index]
table["group"] = pd.Series(roots, index=table.index).map({r: k for k, r in enumerate(sorted(set(roots)))})
assert table.groupby("group")["corpus"].nunique().eq(1).all()
sizes = table.groupby(["corpus", "group"]).size()
print(f"Bach: {sum(1 for f in BACH if zahn.get(f))} chorales with a Zahn number; {len(close)} pairs of close melodies.")
for name, largest in sizes.groupby("corpus").idxmax().items():
    members = table.index[table["group"] == largest[1]]
    label = ", ".join(sorted({tune[f] for f in members if tune.get(f)})) if name == "bach" else record(members[0], "OPR")
    print(f"Largest {name} group: {len(members)} files; {label}.")
sizes.groupby("corpus").describe()[["count", "mean", "max"]].rename(columns={"count": "groups", "mean": "files per group", "max": "largest"})

**Result**
- Bach: 305 chorales with a Zahn number, 271 pairs of close melodies: 181 groups, 117 of a single chorale. The largest holds 13 chorales on *Befiehl du deine Wege* (melody 1).
- Palestrina: 101 groups, one per mass, of 6 to 25 movements.

## Folds

- **Use:** cross-validation by group; ten folds, each tested with a model counted on the other nine.
- **Why ten:** [Kohavi (1995)](https://mlanthology.org/ijcai/1995/kohavi1995ijcai-study); nine tenths of the corpus stay for counting.
- **Assignment:** largest group first, to the fold with the fewest notes, per corpus. No randomness, so no seed: scikit-learn's `GroupKFold` without shuffling (version 1.6), sized by notes.
- **Complete corpus:** all folds together; it builds the hypergraph for the analyses and for generation on new melodies.

In [ ]:
FOLDS = 10
version_rows = []   # one row per version of every file that stays
for f in table.index:
    spans = versions.get(f, [skips[f]])
    for k, span in enumerate(spans, start=1):
        version_rows.append({"file": f, "version": k, "skip": ";".join(f"{s:g}-{e:g}" for s, e in span), "weight": weight.at[f] / len(spans)})
corpus = pd.DataFrame(version_rows).join(table, on="file")
corpus["weighted_notes"] = corpus["weight"] * corpus["voice_notes"]

fold = {}
for name, group in corpus.groupby("corpus"):
    load = np.zeros(FOLDS)
    for g, notes in group.groupby("group")["weighted_notes"].sum().sort_values(ascending=False, kind="stable").items():
        fold[g] = int(np.argmin(load))
        load[fold[g]] += notes
corpus["fold"] = corpus["group"].map(fold)
assert corpus.groupby("group")["fold"].nunique().eq(1).all()
assert np.allclose(corpus.groupby("copy_group")["weight"].sum(), 1)   # every piece weighs one, however many versions it has
summary = corpus.groupby(["corpus", "fold"]).agg(groups=("group", "nunique"), files=("file", "nunique"), weight=("weight", "sum"), notes=("weighted_notes", "sum")).round(1)
summary.unstack(0)

**Result**
- Bach: 17 to 19 groups, 33 to 40 files, 7,784 to 7,909 weighted notes per fold.
- Palestrina: 10 or 11 masses, 99 to 132 movements, 67,270 to 70,043 notes per fold.
- Folds are close in notes; their file counts vary with the size of the groups.

## The corpus table

One row per version of a file: file, version, spans left out (quarter notes from the start), weight, group, fold, then the columns of 01_corpus.

In [ ]:
COLUMNS = ["file", "version", "skip", "weight", "group", "fold", "copy_group", "corpus", "voices", "lines", "voice_parts", "voice_notes",
           "key_signature", "final", "tonal_type", "key", "key_source", "shift"]
corpus[COLUMNS].to_csv(INTERIM / "02_corpus.csv", index=False)   # read by the notebook that builds the hypergraph
corpus.groupby("corpus").agg(rows=("file", "size"), files=("file", "nunique"), weight=("weight", "sum"), groups=("group", "nunique"), notes=("weighted_notes", "sum")).round(1)

**Result:** Bach, 359 rows for 352 chorales, weighing 333 pieces; Palestrina, 1,181 rows for 1,179 movements, weighing 1,178.